In [33]:
import os
import json
import time
from tqdm.auto import tqdm
import concurrent.futures
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_naver import ClovaXEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

## 1. Load json file

In [4]:
def load_json_as_documents(file_path):
    """
    JSON 파일을 읽어 LangChain Document 객체 리스트로 변환합니다.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    docs = []

    for chunk in data:
        chunk_id = chunk["chunk_id"]
        term = chunk["용어"]
        description = chunk["설명"]
        metadata = chunk.get("metadata", {})

        page = metadata.get("page")
        source = metadata.get("source", "경제금융용어700선")

        text = f"{term}\n\n{description}"

        doc = Document(
            page_content=text,
            metadata={
                "chunk_id": chunk_id,
                "term": term,
                "source": source,
                "page": page
            }
        )

        docs.append(doc)
    
    return docs

In [5]:
file_path = "data/" 
docs = load_json_as_documents(file_path+"final_chunk.json")
print(f"로드된 문서 개수: {len(docs)}")

로드된 문서 개수: 700


In [6]:
docs[0:10]

[Document(metadata={'chunk_id': 'econ_0001', 'term': '가계부실위험지수(HDRI)', 'source': 'data/2020_경제금융용어 700선_게시.pdf', 'page': 18}, page_content='가계부실위험지수(HDRI)\n\n가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상 환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다.'),
 Document(metadata={'chunk_id': 'econ_0002', 'term': '가계수지', 'source': 'data/2020_경제금융용어 700선_게시.pdf', 'page': 18}, page_content="가계수지\n\n가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 흑자를 냈다면 그 가정은 벌어들인 수입 일부만을 사용했다는 것을 의미하며, 적자를 냈다면 수입 외에 빚을 추가로 얻어 사용한 것이라고 보아야 한다. 우리나라는 통계청에 서 가계의 수입과 지출을 조사하여 국민의 소득수준 및 생활실태를 파악하

## 2. VectorDB Embedding

### Chroma DB 생성

In [ ]:
# OpenAI용 Chroma DB 생성
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore_openai = Chroma(
    collection_name="docs_openai",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=openai_embeddings,
    persist_directory="./chroma_openai" # 별도 경로 설정
)

# CLOVA용 Chroma DB 생성
clova_embeddings = ClovaXEmbeddings(model="bge-m3")
vectorstore_clova = Chroma(
    collection_name="docs_clova",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=clova_embeddings,
    persist_directory="./chroma_clova"
)

### Embedding

In [ ]:
def process_vectorstore(name, vectorstore, documents, batch_size=100, pos=0):
    total_chunks = len(documents)
    all_ids = [doc.metadata["chunk_id"] for doc in documents]
    
    pbar = tqdm(total=total_chunks, desc=f"[{name}]", position=pos, leave=True)
    
    for i in range(0, total_chunks, batch_size):
        batch_docs = documents[i : i + batch_size]
        batch_ids = all_ids[i : i + batch_size]
        
        # 성공할 때까지 반복하는 재시도 로직 (최대 3번)
        retry_count = 0
        while retry_count < 3:
            try:
                vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
                pbar.update(len(batch_docs))
                
                # [핵심] 성공 후 다음 배치를 보내기 전 약간의 휴식 (특히 CLOVA를 위해)
                if name == "CLOVA":
                    time.sleep(2.0)  # CLOVA는 조금 더 넉넉히 2초
                else:
                    time.sleep(0.5)  # OpenAI는 0.5초 정도로 충분
                break 
                
            except Exception as e:
                if "429" in str(e):
                    retry_count += 1
                    tqdm.write(f"[{name}] 429 에러 감지. {retry_count}차 재시도 중... (5초 대기)")
                    time.sleep(5)  # 429 에러 시에는 더 길게 쉽니다.
                else:
                    tqdm.write(f"[{name}] 기타 에러 발생: {e}")
                    retry_count = 3 # 다른 에러는 중단
            
    pbar.close()


if __name__ == "__main__":
    # --- 실행부 ---
    # 미리 정의된 vectorstore 객체들 (OpenAI, CLOVA)
    tasks = [
        ("OpenAI", vectorstore_openai, docs, 0),
        ("CLOVA", vectorstore_clova, docs, 1)
    ]

    print("병렬 임베딩 시작 (ID 지정 및 배치 처리)...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [
            executor.submit(process_vectorstore, name, vs, docs, 100, p) 
            for name, vs, docs, p in tasks
        ]
        concurrent.futures.wait(futures)

    print("\n모든 작업이 완료되었습니다.")

병렬 임베딩 시작 (ID 지정 및 배치 처리)...


[CLOVA]: 100%|██████████| 700/700 [01:18<00:00,  8.92it/s]


모든 작업이 완료되었습니다.


In [15]:
# # 임베딩 모델 설정
# embeddings = OpenAIEmbeddings(
#     model="text-embedding-3-small"
# )

# # Chroma DB 로컬 저장 경로
# db_path = "./finance_word_db"

# # 벡터 DB 생성 및 코사인 유사도 설정
# vectorstore = Chroma(
#     persist_directory=db_path,
#     embedding_function=embeddings,
#     collection_metadata={"hnsw:space": "cosine"} # 코사인 유사도 설정
# )

# ids = [doc.metadata["chunk_id"] for doc in docs]

# vectorstore.add_documents(
#     documents=docs,
#     ids=ids
# )

## 3. Retrieval

In [22]:
query_1 = "비트코인이 뭐야?"
query_2 = "금리가 오르면 가계부실위험지수는 어떻게 되나요?"
query_3 = "DSR 계산 시 금리가 미치는 영향과 가계부실위험지수의 관계를 설명해줘"
query_4 = "KIKO 옵션이 수출기업에 미치는 위험은 무엇인가요?"

### Similarity Search

In [25]:
# Similarity Search (유사도 검색)
# 단순히 질문과 가장 유사한 문서 조각들을 리스트 형태로 반환합니다.
print(f"\n질문: '{query_1}'")

print(f"\n=== OpenAI Similarity Search 결과 ===")
res_openai = vectorstore_openai.similarity_search(query_1, k=3)
for i, doc in enumerate(res_openai):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)

print(f"\n=== Clova Similarity Search 결과 ===")
res_clova = vectorstore_clova.similarity_search(query_1, k=3)
for i, doc in enumerate(res_clova):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)


질문: '비트코인이 뭐야?'

=== OpenAI Similarity Search 결과 ===
[1] 용어: 비트코인
내용 요약: 비트코인

비트코인(bitcoin)은 가상통화(암호통화)이자 디지털 지급시스템이다. 비트코인 시스템 은 중앙 저장소 또는 단일 관리자가 없기 때문에 최초의 탈중앙화된 디지털통화라고...
------------------------------
[2] 용어: 가상통화
내용 요약: 가상통화

가상통화(virtual currency)는 중앙은행이나 금융기관이 아닌 민간에서 블록체인을 기반 기술로 하여 발행･유통되는 ‘가치의 전자적 표시’(digital repr...
------------------------------
[3] 용어: 전자화폐
내용 요약: 전자화폐

IC카드 등에 화폐가치를 저장하였다가 상품 등의 구매에 사용할 수 있는 전자지급수단 으로서 범용성과 환금성을 갖춘 것을 말한다. 전자화폐(electronic money)...
------------------------------

=== Clova Similarity Search 결과 ===
[1] 용어: 비트코인
내용 요약: 비트코인

비트코인(bitcoin)은 가상통화(암호통화)이자 디지털 지급시스템이다. 비트코인 시스템 은 중앙 저장소 또는 단일 관리자가 없기 때문에 최초의 탈중앙화된 디지털통화라고...
------------------------------
[2] 용어: 가상통화
내용 요약: 가상통화

가상통화(virtual currency)는 중앙은행이나 금융기관이 아닌 민간에서 블록체인을 기반 기술로 하여 발행･유통되는 ‘가치의 전자적 표시’(digital repr...
------------------------------
[3] 용어: 가상통화공개(ICO)
내용 요약: 가상통화공개(ICO)

가상통화(ICO; Initial Coin Offering) 공개는 주로 혁신적인 신생기업(startup)이 암호 화화폐(cryptocurrency) 또는 디...
----

In [26]:
print(f"\n질문: '{query_3}'")

print(f"\n=== OpenAI Similarity Search 결과 ===")
res_openai = vectorstore_openai.similarity_search(query_3, k=3)
for i, doc in enumerate(res_openai):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)

print(f"\n=== Clova Similarity Search 결과 ===")
res_clova = vectorstore_clova.similarity_search(query_3, k=3)
for i, doc in enumerate(res_clova):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)


질문: 'DSR 계산 시 금리가 미치는 영향과 가계부실위험지수의 관계를 설명해줘'

=== OpenAI Similarity Search 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는...
------------------------------
[2] 용어: 시스템 리스크
내용 요약: 시스템 리스크

금융시스템 일부 또는 전부의 장애로 금융 기능이 정상적으로 수행되지 못함에 따라 실물경제에 심각한 부정적 파급효과를 미칠 수 있는 위험을 의미한다. G10(2001...
------------------------------
[3] 용어: 꼬리위험
내용 요약: 꼬리위험

경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위...
------------------------------

=== Clova Similarity Search 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는...
------------------------------
[2] 용어: 총부채원리금상환비율(DSR)
내용 요약: 총부채원리금상환비율(DSR)

차주의 상환능력 대비 원리금상환부담을 나타내는 지표로서, 차주가 보유한 모든 대출의 연간 원리금상환액을 연간소득으로 나누어 산출된다. 대출에는 마이너...
------------------------------
[3] 용어: 부채비율
내용 요약: 부채비율

부채를 자기자본으로 나눈 비율로 타인자본(부채)과 자기자본간의 관계를 나타내는 대

In [27]:
print(f"\n질문: '{query_4}'")

print(f"\n=== OpenAI Similarity Search 결과 ===")
res_openai = vectorstore_openai.similarity_search(query_4, k=3)
for i, doc in enumerate(res_openai):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)

print(f"\n=== Clova Similarity Search 결과 ===")
res_clova = vectorstore_clova.similarity_search(query_4, k=3)
for i, doc in enumerate(res_clova):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)


질문: 'KIKO 옵션이 수출기업에 미치는 위험은 무엇인가요?'

=== OpenAI Similarity Search 결과 ===
[1] 용어: KIKO
내용 요약: KIKO

환율이 특정 구간(barrier)에 도달하는 경우 옵션이 발효(KI; Knock-In)되거나 소멸 (KO; Knock-Out)되는 조건이 부과된 비정형적인 통화옵션 거래...
------------------------------
[2] 용어: 수출보험
내용 요약: 수출보험

상품수출 시 수출업자가 수입업자의 신용위험, 수입국의 정치상황 등으로 수출대금을 회수하지 못할 위험으로부터 수출업자를 보호하기 위한 보험이다. 수출보험의 운영을 위해 한...
------------------------------
[3] 용어: 꼬리위험
내용 요약: 꼬리위험

경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위...
------------------------------

=== Clova Similarity Search 결과 ===
[1] 용어: KIKO
내용 요약: KIKO

환율이 특정 구간(barrier)에 도달하는 경우 옵션이 발효(KI; Knock-In)되거나 소멸 (KO; Knock-Out)되는 조건이 부과된 비정형적인 통화옵션 거래...
------------------------------
[2] 용어: 환리스크
내용 요약: 환리스크

장래의 예상하지 못한 환율변동으로 인하여 보유한 외화표시 순자산(자산-부채) 또는 현금흐름의 가치가 변동될 수 있는 불확실성을 의미한다. 환리스크는 기본적으로 외환포 지...
------------------------------
[3] 용어: 가상통화공개(ICO)
내용 요약: 가상통화공개(ICO)

가상통화(ICO; Initial Coin Offering) 공개는 주로 혁신적인 신생기업(startup)이 암호 화화폐(cryptocu

### MMR (Maximal Marginal Relevance search)

In [28]:
# 리트리버는 나중에 LLM(ChatGPT 등)과 연결하기 위한 인터페이스입니다.
# score_threshold 등을 설정하여 좀 더 정교한 검색이 가능합니다.

retriever_openai = vectorstore_openai.as_retriever(
    search_type="mmr", # 기본 'similarity' 대신 'mmr' 사용
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7 # 0에 가까울수록 다양성 중시, 1에 가까울수록 유사도 중시
    }
)

retriever_clova = vectorstore_clova.as_retriever(
    search_type="mmr", # 기본 'similarity' 대신 'mmr' 사용
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7 # 0에 가까울수록 다양성 중시, 1에 가까울수록 유사도 중시
    }
)

In [29]:
print(f"\n질문: '{query_2}'")

print(f"\n=== OpenAI MMR Retriever 결과 ===")
res_openai = retriever_openai.invoke(query_2)
for i, doc in enumerate(res_openai):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)

print(f"\n=== Clova MMR Retriever 결과 ===")
res_clova = retriever_clova.invoke(query_2)
for i, doc in enumerate(res_clova):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)


질문: '금리가 오르면 가계부실위험지수는 어떻게 되나요?'

=== OpenAI MMR Retriever 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는...
------------------------------
[2] 용어: 꼬리위험
내용 요약: 꼬리위험

경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위...
------------------------------
[3] 용어: 금리스왑
내용 요약: 금리스왑

금리스왑(IRS; Interest Rate Swaps)이란 금리변동위험 헤지 및 차입비용 절감 등을 위하여 거래당사자간에 원금교환 없이 정기적(3개월)으로 변동금리(91...
------------------------------
[4] 용어: 신용위험(신용리스크)
내용 요약: 신용위험(신용리스크)

채권･채무관계에서 채무자의 채무불이행, 이행거부 또는 신용도 하락 등으로 인해 손실이 발생할 가능성을 의미한다. 바젤 자본규제에서는 주어진 신뢰 수준(99....
------------------------------
[5] 용어: 고정금리
내용 요약: 고정금리

고정금리란 최초 약정한 금리가 만기때까지 그대로 유지되는 금리를 의미하며 변동금 리란 일정 주기별로 시장 금리를 반영하여 약정금리가 변동하는 금리를 의미한다. 예를 들어...
------------------------------

=== Clova MMR Retriever 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의

In [30]:
print(f"\n질문: '{query_3}'")

print(f"\n=== OpenAI MMR Retriever 결과 ===")
res_openai = retriever_openai.invoke(query_3)
for i, doc in enumerate(res_openai):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)

print(f"\n=== Clova MMR Retriever 결과 ===")
res_clova = retriever_clova.invoke(query_3)
for i, doc in enumerate(res_clova):
    print(f"[{i+1}] 용어: {doc.metadata.get('term')}")
    print(f"내용 요약: {doc.page_content[:100]}...") # 용어 설명은 앞부분만 출력
    print("-" * 30)


질문: 'DSR 계산 시 금리가 미치는 영향과 가계부실위험지수의 관계를 설명해줘'

=== OpenAI MMR Retriever 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는...
------------------------------
[2] 용어: 꼬리위험
내용 요약: 꼬리위험

경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위...
------------------------------
[3] 용어: 고정금리
내용 요약: 고정금리

고정금리란 최초 약정한 금리가 만기때까지 그대로 유지되는 금리를 의미하며 변동금 리란 일정 주기별로 시장 금리를 반영하여 약정금리가 변동하는 금리를 의미한다. 예를 들어...
------------------------------
[4] 용어: 유동성리스크
내용 요약: 유동성리스크

거래 일방이 일시적인 자금부족으로 인해 정해진 결제 시점에서 결제의무를 이행하지 못함에 따라 거래상대방의 자금조달 계획 등에 악영향을 미치게 되는 위험을 말한다. 유...
------------------------------
[5] 용어: 감응도계수
내용 요약: 감응도계수

󰡔전방연쇄효과󰡕 참조...
------------------------------

=== Clova MMR Retriever 결과 ===
[1] 용어: 가계부실위험지수(HDRI)
내용 요약: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는...
------------------------------
[2] 용어: 총부채원리금상환비

## 4. Prompt, LLM, 리트리버 연결

In [31]:
# 1. 모델 설정
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0,      # 금융 정보이므로 창의성보다는 정확도를 위해 0 설정
    max_tokens=500      # 출력 길이 제한으로 비용 절감
)

# 2. 프롬프트 설정 (금융 전문가 컨셉)
template = """
[Role]
You are a logical Financial Analyst.

[Instruction]
Answer the question using the provided Context. 
If the direct relationship is missing, logically infer the impact using the definitions provided (e.g., DSR, HDRI). 
Explain the 'Why' behind your reasoning based on economic principles.

[Context]
{context}

[Question]
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)

# 3. 체인 생성 (LCEL 방식)
# retriever는 앞서 만든 vectorstore.as_retriever()를 사용합니다.
rag_chain_with_source = RunnableParallel(
    {"context": retriever_clova, "question": RunnablePassthrough()}
).assign(answer=prompt | llm | StrOutputParser())

# 4. 실행
response = rag_chain_with_source.invoke(query_3)

# 5. 결과 출력
print(f"AI 답변:\n{response['answer']}\n")

print("참조한 문서 리스트:")
for i, doc in enumerate(response['context']):
    print(f"[{i+1}] {doc.metadata.get('title')} (출처: {doc.metadata.get('source')}, {doc.metadata.get('page')}p)")
    # 문서 내용이 너무 길면 앞부분만 출력
    print(f"내용: {doc.page_content[:150]}...")
    print("-" * 30)

AI 답변:
DSR(총부채원리금상환비율)은 차주가 보유한 모든 대출의 연간 원리금상환액을 연간소득으로 나누어 산출되는 지표입니다. 금리는 DSR 계산에 직접적인 영향을 미치며, 이는 차주의 원리금상환부담에 영향을 주기 때문입니다. 

1. **금리의 영향**: 금리가 상승하면 대출의 이자 부담이 증가하게 됩니다. 이는 차주가 매년 상환해야 하는 원리금의 총액을 증가시키며, 결과적으로 DSR이 높아지게 됩니다. 반대로 금리가 하락하면 이자 부담이 줄어들어 DSR이 낮아질 수 있습니다. 

2. **가계부실위험지수(HDRI)와의 관계**: HDRI는 DSR과 DTA(부채/자산비율)를 결합하여 가계의 부실위험을 평가하는 지표입니다. DSR이 높아지면 가계의 채무상환능력이 약화되므로, HDRI도 상승하게 됩니다. 이는 가계가 소득 측면에서 취약해질 가능성을 나타내며, DSR이 40%를 초과할 경우 위험가구로 분류될 수 있습니다. 

3. **경제적 원리**: 경제적으로 볼 때, 금리는 자본의 비용을 나타내며, 이는 소비자와 기업의 대출 수요에 영향을 미칩니다. 금리가 높아지면 대출이 줄어들고 소비가 감소할 수 있으며, 이는 경제 전반에 부정적인 영향을 미칠 수 있습니다. 따라서 DSR이 높아지고 HDRI가 상승하는 것은 가계의 재정적 안정성을 저하시켜 경제 전반에 부정적인 영향을 미칠 수 있습니다.

결론적으로, 금리는 DSR에 직접적인 영향을 미치며, 이는 가계부실위험지수(HDRI)와 밀접한 관계를 형성합니다. 금리가 상승하면 DSR이 높아지고, 이는 가계의 부실위험을 증가시켜 경제적 불안정을 초래할 수 있습니다.

참조한 문서 리스트:
[1] None (출처: data/2020_경제금융용어 700선_게시.pdf, 18p)
내용: 가계부실위험지수(HDRI)

가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상 환비율(DSR; Debt Service Ratio

In [ ]:
# template = """아래 제공된 문맥(Context)만을 사용하여 질문에 답하세요. 
# 답을 모른다면 모른다고 하고, 억지로 답변을 만들어내지 마세요.

# Context:
# {context}

# Question: {question}
# """

In [ ]:
# 질문: 금리가 오르면 가계부실위험지수는 어떻게 되나요?

# AI 답변:
# 제공된 문맥에서는 금리가 오르면 가계부실위험지수(HDRI)에 대한 직접적인 언급이 없습니다. 따라서 금리 인상이 가계부실위험지수에 미치는 영향을 정확히 알 수 없습니다.

# 참조한 문서 리스트:
# [1] 가계부실위험지수(HDRI) (출처: data/2020_경제금융용어 700선_게시.pdf, 18p)
# 내용: 용어: 가계부실위험지수(HDRI)
# 설명: 가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려해 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상 환비율(DSR; Debt Service Ratio)와 자산 측면에서 ...
# ------------------------------
# [2] 꼬리위험 (출처: data/2020_경제금융용어 700선_게시.pdf, 91p)
# 내용: 용어: 꼬리위험
# 설명: 경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위험이다. 주가, 환율 등 시장데이터에서 분포의 꼬리 부분이 두터워지는 경우(f...
# ------------------------------
# [3] 예상손실 (출처: data/2020_경제금융용어 700선_게시.pdf, 212p)
# 내용: 용어: 예상손실
# 설명: 예상손실은 현재 시점부터 일정 기간동안 발생할 것으로 예상되는 손실의 평균금액을 의미하며 일반적으로 자산별로 발생 가능한 손실액에 발생 확률을 곱해 산출한다. 바젤 자본규제에서는 신용리스크로 인한 총손실을 99.9% 신뢰 수준에서 1년 동안 발생...
# ------------------------------

In [ ]:
# 질문: 금리가 오르면 가계부실위험지수는 어떻게 되나요?

# AI 답변:
# 금리가 오르면 가계부실위험지수(HDRI)는 일반적으로 상승할 가능성이 높습니다. 그 이유는 다음과 같은 경제 원칙에 기반합니다.

# 1. **DSR의 증가**: 가계부실위험지수는 가계의 채무상환능력을 평가하는 원리금상환비율(DSR)과 자산 측면에서 평가하는 부채/자산비율(DTA)을 결합하여 산출됩니다. 금리가 상승하면 대출의 이자 비용이 증가하게 되어, 가계의 원리금 상환 부담이 커집니다. 이로 인해 DSR이 증가하게 되고, 이는 HDRI의 상승으로 이어집니다.

# 2. **소득의 상대적 감소**: 금리가 오르면 대출 이자 부담이 증가하여 가계의 가처분 소득이 줄어들 수 있습니다. 이는 가계가 소비를 줄이게 만들고, 경제 전반에 부정적인 영향을 미칠 수 있습니다. 소득이 줄어들면 채무 상환 능력이 더욱 악화되어 DSR이 더욱 증가하게 됩니다.

# 3. **자산 가치의 하락**: 금리가 상승하면 일반적으로 자산 가격, 특히 부동산 가격이 하락하는 경향이 있습니다. 자산 가치가 하락하면 DTA가 증가하게 되어, 이는 다시 HDRI의 상승으로 이어질 수 있습니다. 자산이 줄어들면 가계의 재정적 안정성이 더욱 취약해지기 때문입니다.

# 결론적으로, 금리가 오르면 가계부실위험지수(HDRI)는 DSR의 증가와 DTA의 변화로 인해 상승할 가능성이 높습니다. 이는 가계의 채무상환능력이 악화되고, 재정적 안정성이 저하되기 때문입니다.

# 참조한 문서 리스트:
# [1] 가계부실위험지수(HDRI) (출처: data/2020_경제금융용어 700선_게시.pdf, 18p)
# 내용: 용어: 가계부실위험지수(HDRI)
# 설명: 가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려해 가계부채의 부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상 환비율(DSR; Debt Service Ratio)와 자산 측면에서 ...
# ------------------------------
# [2] None (출처: data/2020_경제금융용어 700선_게시.pdf, 91p)
# 내용: 꼬리위험

# 경제에 미치는 충격의 확률분포곡선이 종(鐘) 모양이라고 가정한다면 양극단 꼬리부 분의 발생 가능성은 매우 낮지만 일단 발생하면 경제 전체에 지대한 영향을 줄 수 있는 위험이다. 주가, 환율 등 시장데이터에서 분포의 꼬리 부분이 두터워지는 경우(fat tail...
# ------------------------------
# [3] None (출처: data/2020_경제금융용어 700선_게시.pdf, 71p)
# 내용: 금리스왑

# 금리스왑(IRS; Interest Rate Swaps)란 금리변동위험 헤지 및 차입비용 절감 등을 위해 거래당사자간에 원금교환 없이 정기적(3개월)로 변동금리(91일물 CD금리)와 고정금리(IRS금리)를 교환하는 거래를 말한다. 이때 고정금리를 지급하는 대신...
# ...
# 내용: 고정금리

# 고정금리란 최초 약정한 금리가 만기때까지 그대로 유지되는 금리를 의미하며 변동금 리란 일정 주기별로 시장 금리를 반영해 약정금리가 변동하는 금리를 의미한다. 예를 들어 만기 1년, 약정금리는 4%의 고정금리라면 약정기간 1년 동안 시장금리가 어떻게 변하더라도...
# ------------------------------

In [ ]:
# 질문: 국민소득, 국민총소득(GNI)과의 관계는?

# AI 답변:
# 국민소득(NI)과 국민총소득(GNI) 간의 관계는 다음과 같이 설명할 수 있습니다.

# 국민소득(NI)은 한 나라의 가계, 기업, 정부 등 모든 경제주체가 일정 기간 동안 생산한 재화와 용역의 가치를 화폐 단위로 평가하여 합산한 것입니다. 이는 국민총소득(GNI)과 밀접한 관계가 있으며, GNI는 국민이 생산활동에 참여한 대가로 받은 소득의 합계로 정의됩니다. GNI는 외국으로부터 받은 소득(국외수취 요소소득)을 포함하고, 외국인에게 지급한 소득(국외지급 요소소득)은 제외합니다.

# 따라서, 국민총소득(GNI)은 국민소득(NI)에서 감가상각과 생산 및 수입세를 조정하여 산출할 수 있습니다. 즉, 국민소득은 GNI에서 감가상각과 생산 및 수입세를 차감한 값으로 정의됩니다. 

# 이러한 관계는 경제의 순환 구조를 반영합니다. 기업이 생산한 재화와 서비스는 가계의 소비를 통해 소득으로 이어지며, 이 과정에서 발생하는 소득은 다시 기업의 생산으로 돌아오는 순환 구조를 형성합니다. 따라서, 국민소득(NI)과 국민총소득(GNI)은 서로 다른 측면에서 경제 활동을 측정하지만, 본질적으로는 동일한 경제적 활동을 반영하고 있습니다.

# 결론적으로, 국민소득(NI)과 국민총소득(GNI)은 서로 연결되어 있으며, GNI는 NI의 확장된 개념으로 외국과의 경제적 상호작용을 포함하는 지표입니다. 이러한 관계는 경제의 생산, 지출, 분배의 세 가지 관점에서 국민소득이 항상 동일해야 한다는 '국민소득 3면 등가의 법칙'에 의해 뒷받침됩니다.

# 참조한 문서 리스트:
# [1] None (출처: data/2020_경제금융용어 700선_게시.pdf, 59p)
# 내용: 국민총소득(GNI)

# 국민총소득(GNI; Gross National Income)는 한 나라의 국민이 생산활동에 참여한 대가로 받은 소득의 합계로서 외국으로부터 국민(거주자)가 받은 소득(국외수취 요소소 득)은 포함되고 국내총생산 중에서 외국인에게 지급한 소득(국외지급...
# ------------------------------
# [2] 국민총소득(GNI) (출처: data/2020_경제금융용어 700선_게시.pdf, 59p)
# 내용: 용어: 국민총소득(GNI)
# 설명: 국민총소득(GNI; Gross National Income)는 한 나라의 국민이 생산활동에 참여한 대가로 받은 소득의 합계로서 외국으로부터 국민(거주자)가 받은 소득(국외수취 요소소 득)은 포함되고 국내총생산 중에서 외국인에게 지급한 ...
# ------------------------------
# [3] None (출처: data/2020_경제금융용어 700선_게시.pdf, 57p)
# 내용: 국민소득

# 국민소득(NI; National Income) 이란 넓은 의미로 볼 때 한 나라 안에 있는 가계, 기업, 정부 등의 모든 경제주체가 일정 기간에 생산한 재화와 용역의 가치를 화폐단위로 평가해 합산한 것으로 흔히 국민총소득으로 불리고 있다. 좁은 의미의 국민소...
# ------------------------------
# [4] None (출처: data/2020_경제금융용어 700선_게시.pdf, 58p)
# 내용: 국민소득 3면 등가의 법칙

# 국민소득이란 한 국가에 있는 가계, 기업, 정부 등 모든 경제주체가 일정 기간(보통 1년)에 새로이 생산한 재화 및 서비스의 가치를 금액으로 평가해 더한 총합을 의미한 다. 즉, 한 국가의 국민 전체가 일정 기간에 새로 벌어들인 소득의 총합...
# ------------------------------
# [5] None (출처: data/2020_경제금융용어 700선_게시.pdf, 35p)
# 내용: 경제후생지표

# 복지지표로서 한계성을 갖는 국민총소득(GNI)를 보완하기 위해 미국의 노드하우스 (W. Nordhaus)와 토빈(J. Tobin)가 제안한 새로운 지표를 말한다. 현재 주요 지표로 활용 중인 국민총소득은 국민 복지에 영향을 미치는 많은 요인(주부의 가사노...
# ------------------------------